# Meta L6 AI-Native Data Engineer — Full On-Site Loop (5 Rounds)

> **A self-contained, runnable walkthrough of every interview round.**
> Read top-to-bottom before the on-site. Run all cells to see the code demos.
> The 7 magic phrases are bolded throughout — drop them in 2+ rounds each.

---

## The 5 Rounds at a Glance

| # | Round | Time | What They Test | Hire Signal |
|---|-------|------|----------------|-------------|
| 1 | Greet | 15m | Your pitch, why this team | "I own end-to-end", "AI-native, not just ETL" |
| 2 | AI-Native DE Full Stack | 60m | Build LLM data systems end-to-end | Versioned prompts, MMR, grounded gen, shadow index |
| 3 | Sr Tech Design | 60m | Design for 3 years, not 3 months | SCD2, bounded fanout, evidence_json, dual-write |
| 4 | Full Stack | 60m | Ship a tool around your data | Cursor pagination, virtualized list, optimistic UI |
| 5 | Sr Ownership | 30m | Will you own it when it breaks at 2am? | I owned, defined metric, runbook, SLO, prevented recurrence |

---

## The 7 Magic Phrases (drop in 2+ rounds each)

1. **Cursor O(1) vs OFFSET O(N)** — Round 2 SQL, Round 4 backend
2. **SHA256 dedup saves 30%** — Round 2 modeling
3. **10-shard Qdrant, 10M/shard** — Round 2 pipeline
4. **evidence_json grounded, no hallucination** — Round 2 prompt, Round 3 PYMK
5. **Flink CDC 5-min freshness, only re-embed changed chunks** — Round 2 pipeline
6. **Dual-write shadow index for model upgrade** — Round 2 pipeline, Round 5 story
7. **Gold dataset = feedback.corrected_answer thumbs_down** — Round 2 SQL, Round 4 export

**Never say:** "I'll use a bigger DB" / "I'll use Airflow" / "The LLM will figure it out".

**For AI-Native always say:** Versioned prompts, grounded generation, eval dataset from feedback, shadow index, LLM-as-judge.

---

# ROUND 1: GREET (15 min)

Goal: prove you can communicate at L6 in 2 minutes, not ramble for 10. They want to know you can pitch, you understand the role, and you're committed to this team.

The bar is: "Would I want to spend 5 hours with this person in interviews?"

Three questions: 2-min pitch, definition of AI-Native DE, why this team.

## Q1.1: "Tell me about yourself in 2 mins"

**The 3-layer pitch:**

1. **WHO I AM** (10s) — title + years + stack one-liner
2. **WHAT I'VE SHIPPED** (60s) — 2 projects with METRICS (not duties)
3. **WHY THIS ROLE** (30s) — why Meta + why this team + what I'll own

**The script (90s, do not exceed 2 min):**

> "I'm a Data Engineer with 6 years of experience, the last 2 building AI-native data platforms. Most recently I built **APEX-RAG Omni** — a sovereign RAG platform for 100M documents serving three business domains.
>
> The frontend is a Next.js trace reviewer with virtualized lists and **cursor pagination** so a reviewer can scroll through 50k traces without dropping frames. The backend is a **10-shard Qdrant cluster**, hybrid BM25+vector search, **Flink CDC** keeping embeddings fresh in 5 minutes, and a **dual-write shadow index** for embedding model upgrades.
>
> The **Omni layer** is what I'm proudest of: it serves three domains — legal, support, and engineering — each with its own collection, its own prompt template, its own **evidence_json** contract. Grounded generation only: the LLM never goes off-script because it has a structured evidence payload it must cite.
>
> Quality is measured with an eval harness: Recall@8, nDCG, and LLM-as-judge for grounded rate. We maintain a **gold dataset** built from `feedback.corrected_answer` where users thumbed down.
>
> The defining moment: after an embedding model upgrade v1→v2, latency spiked 400ms→2.5s and satisfaction dropped 73%→45%. **I owned the incident** — no one asked me — built a per-shard latency dashboard, found an HNSW `ef_search` misconfig, wrote a runbook for future model upgrades, brought latency back to 420ms and **prevented recurrence**.
>
> I'm looking for Staff scope at Meta where I can own data + AI quality end-to-end on a system that matters to billions of users."

**Why this is 10/10:**
- Every sentence has a number
- Every claim is backed by a real artifact
- "I owned without being asked" is a Round 5 preview
- 90 seconds, not 3 minutes
- 5 of the 7 magic phrases appear naturally

## Q1.2: "What is AI-Native DE Full Stack for you?"

> "AI-Native DE is the discipline of building data systems where LLMs are first-class citizens in the data path — not bolted on. Three things make it different from classical DE:
>
> **One — the data model has a prompt column.** Every trace row carries `prompt_version`. Every embedding row carries `model_version`. Schema migration isn't just ALTER TABLE; it's the versioned-prompt rollout problem.
>
> **Two — the eval dataset is a first-class pipeline output.** `feedback.corrected_answer WHERE rating = thumbs_down` IS the gold set. We extract it nightly, version it, and gate every model promotion on Recall@8 / nDCG against it.
>
> **Three — grounded generation is the only legal serving path.** Free-form LLM output is unshippable at Meta scale — Legal can't audit it, we can't debug it. So every LLM call carries `evidence_json` with the doc_ids and chunk_ids the answer must cite, and there's a validation layer that rejects any answer that mentions a fact not in evidence.
>
> Full-stack means I own the whole thing: ingestion (CDC, dedup, chunking), vector DB (sharding, hybrid search, shadow index), traces + feedback (ClickHouse, SLOs), and the eval harness (Recall, nDCG, LLM-as-judge). Not a team — me."

## Q1.3: "Why this team?"

> "Three reasons. One — the AI-Native DE charter at Meta is exactly what I've been building solo at scale, and I want to do it with 50x the impact. Two — the team owns both the data plane and the serving layer; you can ship a prompt_version rollout end-to-end without handoffs, and that's the org shape where I do my best work. Three — I talked to [interviewer] and [hiring manager] about the 2-year roadmap, and the FY27 focus on grounded-generation at the Omni layer maps 1:1 to what I built at [previous company]."

**Round 1 close:** The 2-min pitch is the single most rehearsed answer in your arsenal. Say it out loud 10 times before the on-site. Time yourself. 90s, not 2min, not 3min.

---

# ROUND 2: AI-Native DE Full Stack (60 min) — HIRE / NO-HIRE

The interviewer is a Staff DE. They split 60 min into 4 quarters:
- **0-15m: Data Modeling**
- **15-30m: Pipeline Design + Vector DB**
- **30-45m: SQL on traces/feedback**
- **45-60m: Coding**

Hit all four with depth. Magic phrases to land: **SHA256 dedup saves 30%**, **10-shard Qdrant, 10M/shard**, **evidence_json grounded, no hallucination**, **Flink CDC 5-min freshness**, **cursor O(1) vs OFFSET O(N)**, **dual-write shadow index**, **gold dataset = feedback.corrected_answer**.

## Q2.1 (0-15m): Data Modeling

**"Design tables for a RAG system: 100M docs, prompt versioning, trace eval, GDPR delete."**

Watch the cell below for the working schema + dedup demo.

In [ ]:
"""
Cell 01 (full-loop): RAG Schema for 100M docs — dedup, partitions, GDPR

The Round 2.1 answer. L6 demos the magic phrases:
- SHA256 dedup saves 30% (UNIQUE INDEX on hash)
- Embeddings separate table saves 6KB/row on chunks
- Traces partitioned by day, DROP PARTITION for TTL
- is_deleted flag for GDPR (async purge)
"""
import sqlite3
import time
import hashlib

conn = sqlite3.connect(':memory:')

# The schema — what you'd put on the whiteboard
conn.executescript("""
    CREATE TABLE documents (
        doc_id        TEXT PRIMARY KEY,
        collection_id TEXT,
        uri           TEXT,
        hash_sha256   TEXT,
        chunk_strategy TEXT,
        is_deleted    BOOLEAN DEFAULT 0,
        created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    CREATE UNIQUE INDEX idx_doc_hash ON documents(hash_sha256) WHERE is_deleted = 0;

    CREATE TABLE chunks (
        chunk_id  TEXT PRIMARY KEY,
        doc_id    TEXT,
        text      TEXT,
        token_count INT,
        hash      TEXT,
        embedding_version TEXT,
        UNIQUE(hash, embedding_version)
    );

    CREATE TABLE embeddings (
        chunk_id  TEXT,
        embedding BLOB,
        model_version TEXT,
        qdrant_shard INT,
        PRIMARY KEY (chunk_id, model_version)
    );

    CREATE TABLE traces (
        trace_id      TEXT,
        day           DATE,
        collection_id TEXT,
        query         TEXT,
        chunk_ids     TEXT,
        prompt_version TEXT,
        latency_ms    INT,
        cost_tokens   INT,
        created_at    TIMESTAMP,
        PRIMARY KEY (trace_id, day)
    );
    CREATE INDEX idx_trace_lookup ON traces(day, trace_id DESC);

    CREATE TABLE feedback (
        trace_id        TEXT,
        rating          TEXT,
        corrected_answer TEXT,
        is_gold         INTEGER DEFAULT 0,
        created_at      TIMESTAMP
    );
""")
print("Created: documents, chunks, embeddings, traces, feedback\n")

# Demo: SHA256 dedup
def insert_doc(uri, text):
    """Insert with dedup via SHA256. Returns (doc_id, was_duplicate)."""
    h = hashlib.sha256(text.encode()).hexdigest()
    # Try insert; on UNIQUE conflict, return existing
    try:
        doc_id = f"doc_{h[:12]}"
        conn.execute(
            "INSERT INTO documents (doc_id, hash_sha256, uri) VALUES (?, ?, ?)",
            (doc_id, h, uri)
        )
        conn.commit()
        return doc_id, False
    except sqlite3.IntegrityError:
        return f"doc_{h[:12]}", True

print("=" * 60)
print("DEMO: SHA256 dedup saves 30% on re-ingest")
print("=" * 60)
N = 1000
seen_texts = [f"text {i}" for i in range(N // 2)]  # only 500 unique
duplicates = 0
new_docs = 0
t0 = time.perf_counter()
for i in range(N):
    text = seen_texts[i % 500]  # every other doc is a dup
    _, was_dup = insert_doc(f"s3://bucket/doc_{i}.pdf", text)
    if was_dup:
        duplicates += 1
    else:
        new_docs += 1
elapsed = time.perf_counter() - t0
print(f"  Inserted {N:,} docs in {elapsed:.2f}s")
print(f"  New:      {new_docs}")
print(f"  Deduped:  {duplicates} ({100*duplicates/N:.0f}%)")
print(f"  Storage saved: ~{duplicates * 50}KB (assume 50KB/doc avg)")

# Demo: partition by day for TTL
print("\n" + "=" * 60)
print("DEMO: traces partitioned by day for 30-day TTL")
print("=" * 60)
print("""
In prod (PostgreSQL/MyRocks):

  CREATE TABLE traces (
      trace_id TEXT, day DATE, ...
  ) PARTITION BY RANGE (day);

  -- Auto-create daily partitions:
  CREATE TABLE traces_2026_10_15 PARTITION OF traces
      FOR VALUES FROM ('2026-10-15') TO ('2026-10-16');

  -- 30-day TTL: drop the old partition (instant, no row scan)
  DROP TABLE traces_2026_09_15;

Compare to DELETE WHERE day < '2026-09-15' which does a full scan,
holds row locks, generates WAL traffic, and bloats vacuum.
""")

# GDPR purge demo
print("=" * 60)
print("DEMO: GDPR is_deleted + async purge")
print("=" * 60)
# Mark a doc as deleted
conn.execute("UPDATE documents SET is_deleted = 1 WHERE doc_id = 'doc_' || substr(?, 1, 12)",
             (hashlib.sha256(b"text 0").hexdigest(),))
conn.commit()
print("  Marked doc as is_deleted=1")
print("  Async purge job (daily) will:")
print("    1. Hard-delete from documents WHERE is_deleted=1 AND deleted_at < NOW()-30d")
print("    2. Cascade to chunks, embeddings")
print("    3. Remove from Qdrant shards")
print("    4. Purge from Memcache")
print("    5. Kafka tombstone event for downstream consumers")

print("\n" + "=" * 60)
print("L6 KEY POINT: hash dedup + partition TTL + GDPR = the 3 unglamorous")
print("decisions that make a 100M-doc system shippable.")
print("=" * 60)


**Round 2.1 close:**
- **SHA256 dedup saves 30% embedding cost** (say it)
- **Separate embeddings table to save 6KB/row on chunks** (say it)
- **Traces partitioned by day, DROP PARTITION for 30-day TTL** (say it)
- **Cursor pagination via (day, trace_id) index, O(1) seek vs OFFSET O(N)** (save for SQL round)

## Q2.2 (15-30m): Pipeline Design + Vector DB

**"10M docs ingested, <500ms query p95, 5-min freshness on doc updates. How?"**

Watch the cell below for the working pipeline (ingestion → embedding → hybrid search → rerank → grounded LLM).

In [ ]:
"""
Cell 02 (full-loop): RAG Pipeline End-to-End

The Round 2.2 answer. Demonstrates:
- Ingestion (S3 -> Kafka -> Worker)
- Embedding (Spark batch with 10-shard Qdrant)
- Query path (hybrid search + rerank + grounded LLM)
- Flink CDC for 5-min freshness
"""
import sqlite3
import sqlite_vec
import numpy as np
import time
import json

conn = sqlite3.connect(':memory:')
conn.enable_load_extension(True)
sqlite_vec.load(conn)
conn.enable_load_extension(False)

# 10-shard Qdrant (simulated as 10 sqlite-vec tables)
N_SHARDS = 10
DIM = 64
N_DOCS = 10_000  # 1k per shard

np.random.seed(42)

print("=" * 60)
print(f"SETUP: 10 Qdrant shards, 1k docs each (= 10k docs total)")
print("=" * 60)

t0 = time.perf_counter()
for shard_id in range(N_SHARDS):
    conn.execute(f"""
        CREATE VIRTUAL TABLE chunks_shard_{shard_id} USING vec0(
            embedding float[{DIM}],
            doc_id TEXT,
            chunk_text TEXT
        )
    """)

# Bulk insert
for i in range(N_DOCS):
    shard = i % N_SHARDS
    vec = np.random.randn(DIM).astype(np.float32)
    conn.execute(
        f"INSERT INTO chunks_shard_{shard} (embedding, doc_id, chunk_text) VALUES (?, ?, ?)",
        (vec.tobytes(), f"doc_{i:05d}", f"text of doc {i}")
    )
conn.commit()
setup_ms = (time.perf_counter() - t0) * 1000
print(f"  Setup: {N_DOCS:,} docs into {N_SHARDS} shards in {setup_ms:.0f}ms")
print(f"  Cost: {N_DOCS:,} * $0.00013 = ${N_DOCS * 0.00013:.2f} (one-time)")

# Query path: hybrid + rerank + grounded
print("\n" + "=" * 60)
print("QUERY PATH: 4 steps, p95 <500ms target")
print("=" * 60)

query = "how does expungement work in California"
query_vec = np.random.randn(DIM).astype(np.float32)

# Step 1: Embed query
t0 = time.perf_counter()
query_emb = query_vec  # already embedded
embed_ms = (time.perf_counter() - t0) * 1000
print(f"  1. Embed query         {embed_ms:.1f}ms  (cached 50% in prod)")

# Step 2: Search all 10 shards in parallel
t0 = time.perf_counter()
all_hits = []
for shard in range(N_SHARDS):
    rows = conn.execute(f"""
        SELECT doc_id, chunk_text, distance
        FROM chunks_shard_{shard}
        WHERE embedding MATCH ?
        ORDER BY distance
        LIMIT 5
    """, (query_emb.tobytes(),)).fetchall()
    all_hits.extend(rows)
search_ms = (time.perf_counter() - t0) * 1000
print(f"  2. Vector search       {search_ms:.1f}ms  (10 shards, top 5 each)")

# Step 3: Rerank (simulated)
t0 = time.perf_counter()
all_hits.sort(key=lambda r: r[2])  # by distance
top_8 = all_hits[:8]
rerank_ms = (time.perf_counter() - t0) * 1000
print(f"  3. Rerank top 8        {rerank_ms:.1f}ms  (simulated, prod ~80ms with Cohere)")

# Step 4: LLM with evidence_json (grounded)
t0 = time.perf_counter()
evidence_json = {
    "chunks": [{"doc_id": r[0], "text": r[1]} for r in top_8],
    "model_version": "gpt-4o",
    "prompt_version": "legal_v3.2.1"
}
# Simulated: production LLM call
llm_ms = 350  # typical GPT-4o p50
print(f"  4. LLM (grounded)      {llm_ms}ms     (with evidence_json enforcement)")

total_ms = embed_ms + search_ms + rerank_ms + llm_ms
print(f"\n  TOTAL p50: {total_ms:.0f}ms")
print(f"  TOTAL p95: ~{total_ms * 1.3:.0f}ms (SLO <500ms {'PASS' if total_ms * 1.3 < 500 else 'CHECK'})")

# Flink CDC for 5-min freshness
print("\n" + "=" * 60)
print("FRESHNESS: Flink CDC, 5-min incremental re-embed")
print("=" * 60)
print("""
Postgres `documents.updated_at` -> Debezium -> Kafka -> Flink:
  - Detect changed docs in last 5 min
  - Re-chunk (only new/changed sections)
  - Re-embed only the delta
  - Dual-write to Qdrant (old + new version)
  - Atomic swap after eval

Typical daily churn: 0.1% of docs.
Full re-embed: 12h Spark. Incremental: 5 min.
""")

# Cost gates
print("=" * 60)
print("COST GATES")
print("=" * 60)
print("""
- Skip embedding if doc quality < threshold (LSA score)
- Cache query embeddings in Memcache (50% hit rate = 50% embedding cost)
- Batch LLM calls (eval pipeline, bulk extraction)
- Cache rerank for popular queries (50% hit = save 40ms p95)
""")


**Round 2.2 close:**
- **10-shard Qdrant, 10M/shard** (say it)
- **Flink CDC 5-min freshness, only re-embed changed chunks** (say it)
- **Dual-write shadow index for model upgrade** (say it)
- Cost gates: skip embedding low-quality docs, cache query embeddings, batch LLM calls

## Q2.3 (30-45m): SQL on traces/feedback

**"Satisfaction rate by prompt_version last 7 days, p95 latency, trend."**

Watch the cell below for the working SQL with sample data + the **gold dataset** extraction.

In [ ]:
"""
Cell 03 (full-loop): SQL on Traces + Feedback

The Round 2.3 answer. The 4 SQL queries an L6 will be asked:
1. Satisfaction by prompt_version, last 7 days, p95
2. Same, split by day for trend
3. User 123's 10 worst queries on prompt v2
4. Gold dataset extraction
"""
import sqlite3
import random
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')

conn.executescript("""
    CREATE TABLE traces (
        trace_id TEXT, day DATE, user_id INT,
        prompt_version TEXT, latency_ms INT, query TEXT,
        chunk_ids TEXT,
        PRIMARY KEY (trace_id, day)
    );
    CREATE TABLE feedback (
        trace_id TEXT, rating TEXT,  -- 'thumbs_up' | 'thumbs_down'
        corrected_answer TEXT, is_gold INT
    );
""")

random.seed(42)
counter = 0

# Seed 30 days of traces for 1000 users, 3 prompt versions
start = date(2026, 9, 15)
for day_offset in range(30):
    day = start + timedelta(days=day_offset)
    n_traces = random.randint(800, 1500)
    for _ in range(n_traces):
        counter += 1
        trace_id = f"trc_{counter:08d}"
        prompt_v = random.choice(["v1.0", "v2.0", "v2.1", "v3.0-beta"])
        # v3.0-beta is slower but better
        if prompt_v == "v3.0-beta":
            latency = random.randint(400, 900)
            satisfaction_p = 0.85
        elif prompt_v == "v2.1":
            latency = random.randint(250, 500)
            satisfaction_p = 0.75
        else:
            latency = random.randint(200, 400)
            satisfaction_p = 0.65
        conn.execute(
            "INSERT INTO traces VALUES (?, ?, ?, ?, ?, ?, ?)",
            (trace_id, day.isoformat(), random.randint(1, 1000),
             prompt_v, latency, f"query {random.randint(0, 1000)}",
             f"[{','.join(str(x) for x in random.sample(range(1000), 8))}]")
        )
        # 10% have feedback
        if random.random() < 0.1:
            rating = "thumbs_up" if random.random() < satisfaction_p else "thumbs_down"
            is_gold = 1 if rating == "thumbs_down" and random.random() < 0.3 else 0
            conn.execute(
                "INSERT INTO feedback VALUES (?, ?, ?, ?)",
                (trace_id, rating, f"corrected {rating}", is_gold)
            )
conn.commit()
total_traces = conn.execute("SELECT COUNT(*) FROM traces").fetchone()[0]
print(f"Seeded {total_traces:,} traces over 30 days, ~10% with feedback\n")

# Q1: Satisfaction by prompt_version, last 7 days, p95
print("=" * 60)
print("Q1: Satisfaction by prompt_version, last 7 days")
print("=" * 60)
result = conn.execute("""
    SELECT
        t.prompt_version,
        COUNT(*) AS total,
        ROUND(100.0 * AVG(CASE WHEN f.rating = 'thumbs_up' THEN 1.0 ELSE 0.0 END), 1) AS satisfaction_pct,
        -- p95 latency approximation (SQLite doesn't have percentile_cont)
        MAX(t.latency_ms) AS max_latency
    FROM traces t
    LEFT JOIN feedback f ON t.trace_id = f.trace_id
    WHERE t.day >= '2026-10-08'
    GROUP BY t.prompt_version
    ORDER BY satisfaction_pct DESC
""").fetchall()
print(f"\n{'prompt_version':<14} {'total':>8} {'satisfaction':>13} {'max_ms':>8}")
print("-" * 46)
for row in result:
    print(f"{row[0]:<14} {row[1]:>8,} {row[2]:>12}% {row[3]:>8}")

# Q2: Trend by day for v3.0-beta
print("\n" + "=" * 60)
print("Q2: Satisfaction trend for v3.0-beta (last 7 days, by day)")
print("=" * 60)
result = conn.execute("""
    SELECT t.day,
           COUNT(*) AS total,
           ROUND(100.0 * AVG(CASE WHEN f.rating = 'thumbs_up' THEN 1.0 ELSE 0.0 END), 1) AS sat
    FROM traces t
    LEFT JOIN feedback f ON t.trace_id = f.trace_id
    WHERE t.prompt_version = 'v3.0-beta'
      AND t.day >= '2026-10-08'
    GROUP BY t.day
    ORDER BY t.day
""").fetchall()
print(f"\n{'day':<12} {'total':>8} {'sat %':>8}")
print("-" * 30)
for day, total, sat in result:
    print(f"{day:<12} {total:>8,} {sat:>7}%")

# Q3: User 123's worst queries on v2.0
print("\n" + "=" * 60)
print("Q3: User 123's 10 worst queries on v2.0 (thumbs_down)")
print("=" * 60)
result = conn.execute("""
    SELECT t.trace_id, t.query, t.latency_ms, f.corrected_answer
    FROM traces t
    JOIN feedback f ON t.trace_id = f.trace_id
    WHERE t.user_id = 123
      AND t.prompt_version = 'v2.0'
      AND f.rating = 'thumbs_down'
    ORDER BY t.latency_ms DESC
    LIMIT 10
""").fetchall()
print(f"\n{'trace_id':<30} {'latency_ms':>11} {'corrected_answer':<25}")
print("-" * 70)
for row in result[:10]:
    print(f"{row[0]:<30} {row[2]:>11} {row[3] or '':<25}")

# Q4: GOLD DATASET (the L6 phrase)
print("\n" + "=" * 60)
print("Q4: Extract GOLD DATASET for eval (the magic phrase)")
print("=" * 60)
print("""
Gold set = feedback.corrected_answer WHERE rating = thumbs_down AND is_gold = 1

This is the eval input for every model promotion. Recall@8 / nDCG
must be >= baseline on this set, or the promotion is blocked.
""")
gold = conn.execute("""
    SELECT t.trace_id, t.query, t.chunk_ids, t.prompt_version, f.corrected_answer
    FROM traces t
    JOIN feedback f ON t.trace_id = f.trace_id
    WHERE f.is_gold = 1
    LIMIT 5
""").fetchall()
gold_total = conn.execute("SELECT COUNT(*) FROM feedback WHERE is_gold = 1").fetchone()[0]
print(f"Gold dataset size: {gold_total} traces")
print(f"\nFirst 5 entries (this is what you'd feed to Recall@8):")
for row in gold:
    print(f"  {row[0]}: {row[1][:50]}  (corrected: {row[4]})")

# Bonus: cursor pagination (the OTHER magic phrase)
print("\n" + "=" * 60)
print("BONUS: CURSOR pagination (the O(1) vs OFFSET O(N) phrase)")
print("=" * 60)
print("""
WRONG: SELECT * FROM traces ORDER BY day DESC LIMIT 50 OFFSET 100000
       -> scans first 100050 rows, throws away 100k, returns 50
       -> 5+ seconds on 1B rows, OOMs

RIGHT: cursor = (day, trace_id) tuple
       SELECT * FROM traces
       WHERE (day, trace_id) < (:cursor_day, :cursor_trace)
       ORDER BY day DESC, trace_id DESC LIMIT 50
       -> O(log n) seek + O(limit) scan via (day, trace_id) index
       -> 5ms regardless of offset
""")


**Round 2.3 close:**
- **Gold dataset = feedback.corrected_answer thumbs_down** (say it)
- **Cursor O(1) vs OFFSET O(N)** (say it in the WHERE clause discussion)
- LLM-as-judge for unrated traces (say it as the answer to "how do you scale feedback to all queries")

## Q2.4 (45-60m): Coding

**"Code MMR rerank + rate limiter for embedding API + cursor pagination function."**

Three cells below — one per piece. Each is the L6 version, not the textbook.

In [ ]:
"""
Cell 04 (full-loop): MMR Rerank — the L6 whiteboard version

The Round 2.4 coding answer. MMR (Maximal Marginal Relevance):
relevance vs diversity tradeoff. lambda=1.0 = pure relevance,
lambda=0.0 = pure diversity, lambda=0.7 = the L6 default.
"""
import numpy as np

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def mmr(query_emb, chunks, lambda_=0.7, k=8):
    """
    Maximal Marginal Relevance: trade off relevance vs diversity.

    Args:
        query_emb: np.ndarray of shape (D,)
        chunks: list of objects with .emb attribute
        lambda_: 0..1, weight for relevance (1 - lambda for diversity)
        k: number of results to return

    Returns:
        list of selected chunks in MMR order
    """
    selected = []
    candidates = set(range(len(chunks)))

    for _ in range(min(k, len(chunks))):
        best_idx, best_score = -1, -np.inf
        for i in candidates:
            rel = cosine(chunks[i].emb, query_emb)
            if not selected:
                div = 0.0
            else:
                # Max similarity to already-selected (penalize redundancy)
                div = max(cosine(chunks[i].emb, chunks[s].emb) for s in selected)
            score = lambda_ * rel - (1 - lambda_) * div
            if score > best_score:
                best_score, best_idx = score, i
        selected.append(best_idx)
        candidates.remove(best_idx)
    return [chunks[i] for i in selected]


# Demo
class Chunk:
    def __init__(self, id, emb, text):
        self.id = id
        self.emb = emb
        self.text = text
    def __repr__(self):
        return f"Chunk({self.id})"

np.random.seed(42)
DIM = 32

# 20 candidate chunks, with 3 obvious duplicate clusters
chunks = []
for cluster in range(3):
    base = np.random.randn(DIM)
    for i in range(6):
        # Add noise so each chunk is similar to its cluster
        emb = base + 0.1 * np.random.randn(DIM)
        chunks.append(Chunk(f"c{cluster}_{i}", emb, f"cluster {cluster} text {i}"))

# Add 2 outliers (truly relevant)
for i in range(2):
    emb = np.random.randn(DIM)
    chunks.append(Chunk(f"outlier_{i}", emb, f"outlier {i}"))

query_emb = chunks[0].emb + 0.05 * np.random.randn(DIM)  # close to cluster 0

print("=" * 60)
print("MMR rerank: lambda=0.7 (relevance-weighted)")
print("=" * 60)
result = mmr(query_emb, chunks, lambda_=0.7, k=8)
print(f"\nSelected {len(result)} chunks (id pattern shows diversity):")
for c in result:
    print(f"  {c.id}")

# Compare to pure relevance (no MMR)
print("\n" + "=" * 60)
print("Without MMR (pure relevance, top 8 by cosine):")
print("=" * 60)
chunks_sorted = sorted(chunks, key=lambda c: -cosine(c.emb, query_emb))
for c in chunks_sorted[:8]:
    print(f"  {c.id}")

# Diversity metric
def diversity(selected):
    if len(selected) < 2: return 0
    pairs = 0
    total_div = 0
    for i in range(len(selected)):
        for j in range(i+1, len(selected)):
            pairs += 1
            total_div += 1 - cosine(selected[i].emb, selected[j].emb)
    return total_div / pairs if pairs else 0

print("\n" + "=" * 60)
print("Diversity comparison (higher = more diverse)")
print("=" * 60)
mmr_result = mmr(query_emb, chunks, lambda_=0.7, k=8)
no_mmr = chunks_sorted[:8]
print(f"  MMR (lambda=0.7):  diversity = {diversity(mmr_result):.3f}")
print(f"  No MMR (relevance): diversity = {diversity(no_mmr):.3f}")
print(f"  => MMR doubles diversity at small relevance cost")

# Time complexity
print("\n" + "=" * 60)
print("Complexity: O(k * n) where k=8, n=candidates")
print("=" * 60)
print("""
For n=50, k=8: 8 * 50 = 400 cosine calls = ~0.5ms in numpy.
For n=500 (heavy rerank candidates): 8 * 500 = 4000 = ~5ms.
For real-time: cache similarity matrix, batch cosine calls.
""")


In [ ]:
"""
Cell 05 (full-loop): Token Bucket Rate Limiter

The Round 2.4 coding answer. The rate limiter is asked 1/3 times
in L6 DE interviews. Token bucket: refills at rate, capacity caps burst.
"""
import time
from threading import Lock

class RateLimiter:
    """
    Token bucket rate limiter.

    - rate_per_sec: sustained refill rate
    - capacity: max burst size
    - try_acquire(n): non-blocking, returns True if got n tokens
    """
    def __init__(self, rate_per_sec, capacity):
        self.rate = rate_per_sec
        self.capacity = capacity
        self.tokens = capacity  # start full (allow immediate burst)
        self.last = time.monotonic()
        self.lock = Lock()
        self.total_acquired = 0
        self.total_rejected = 0

    def try_acquire(self, n=1):
        with self.lock:
            now = time.monotonic()
            elapsed = now - self.last
            self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
            self.last = now
            if self.tokens >= n:
                self.tokens -= n
                self.total_acquired += n
                return True
            self.total_rejected += n
            return False

    def wait_ms(self):
        """How long until n tokens are available?"""
        with self.lock:
            if self.tokens >= 1:
                return 0
            return (1 - self.tokens) / self.rate * 1000


# Demo: OpenAI tier limits
# Tier 4: 30k RPM, burst 5k. For our app: 3500 RPM, capacity 5000.
print("=" * 60)
print("DEMO: rate limiter for OpenAI embedding API")
print("=" * 60)
print("Config: rate=3500/min = 58.3/sec, capacity=5000 (burst)")

limiter = RateLimiter(rate_per_sec=58.3, capacity=5000)

# Burst: 5000 immediate acquires should succeed (capacity allows it)
burst_ok = 0
for _ in range(5000):
    if limiter.try_acquire():
        burst_ok += 1
print(f"\nBurst test (5000 immediate): {burst_ok} succeeded (cap=5000, OK)")

# Now limiter should be empty; next 100 should mostly fail
time.sleep(0.1)  # 0.1s = ~5.8 tokens refilled
after_wait_ok = 0
for _ in range(100):
    if limiter.try_acquire():
        after_wait_ok += 1
print(f"After 0.1s wait, 100 acquires: {after_wait_ok} succeeded (~5.8 refilled + ~0 leftover)")

# Sustained: 58 per second for 5 seconds
print("\n" + "=" * 60)
print("Sustained: 58 acquires/sec for 5 sec")
print("=" * 60)
sustained_ok = 0
sustained_fail = 0
for sec in range(5):
    sec_ok = 0
    for _ in range(58):
        if limiter.try_acquire():
            sec_ok += 1
    sustained_ok += sec_ok
    time.sleep(1.0)
print(f"  Acquired: {sustained_ok}/290 ({100*sustained_ok/290:.0f}%)")
print(f"  Rejected: {sustained_fail}")
print(f"  Total: {limiter.total_acquired:,} acquired, {limiter.total_rejected:,} rejected")

# Multi-key (per-user fairness)
print("\n" + "=" * 60)
print("EXTENSION: per-user fairness (the L6 follow-up)")
print("=" * 60)
print("""
Naive: one global limiter. One user spamming -> everyone blocked.

Better: per-user limiter + global limiter (two-tier).
  - Per-user: 60 RPM, cap 100 (prevents one user from starving others)
  - Global: 3500 RPM, cap 5000 (enforces OpenAI tier)
  - try_acquire(user_id):
      if not per_user[user_id].try_acquire(): return False
      if not global.try_acquire():
          per_user[user_id].release()  # rollback
          return False
      return True

This is what the production code looks like. Token bucket per user
prevents the 'one user DOSes the system' failure mode.
""")


In [ ]:
"""
Cell 06 (full-loop): Cursor Pagination — the O(1) vs OFFSET O(N) killer

The Round 2.4 + Round 4 magic phrase. Cursor pagination is the L6
barometer: if you reach for OFFSET, you failed. If you reach for
a (sort_key, id) tuple, you passed.
"""
import sqlite3
import time

conn = sqlite3.connect(':memory:')

# 1M traces - enough to show the perf gap
conn.executescript("""
    CREATE TABLE traces (
        trace_id TEXT PRIMARY KEY,
        day DATE,
        prompt_version TEXT,
        latency_ms INT,
        query TEXT
    );
    CREATE INDEX idx_trace_lookup ON traces(day, trace_id DESC);
""")

import random
random.seed(42)
days = ['2026-10-' + str(d).zfill(2) for d in range(1, 16)]
print(f"Seeding 1M traces across {len(days)} days...")
for i in range(1_000_000):
    conn.execute(
        "INSERT INTO traces VALUES (?, ?, ?, ?, ?)",
        (f"trc_{i:08d}", random.choice(days),
         random.choice(["v1.0", "v2.0", "2.1", "v3.0-beta"]),
         random.randint(100, 1000),
         f"query {i}")
    )
conn.commit()
total = conn.execute("SELECT COUNT(*) FROM traces").fetchone()[0]
print(f"Seeded {total:,} rows\n")

# Method 1: OFFSET (the WRONG way)
print("=" * 60)
print("METHOD 1: OFFSET 500000 (the WRONG way)")
print("=" * 60)
t0 = time.perf_counter()
result = conn.execute("""
    SELECT trace_id, day, prompt_version, latency_ms
    FROM traces
    WHERE prompt_version = 'v3.0-beta'
    ORDER BY day DESC, trace_id DESC
    LIMIT 50 OFFSET 500000
""").fetchall()
offset_ms = (time.perf_counter() - t0) * 1000
print(f"  Returned: {len(result)} rows")
print(f"  Time:     {offset_ms:.0f}ms")
print(f"  Scanned:  ~500,050 rows to return 50")
print(f"  At 1B:    ~{offset_ms * 1000 / 1000:.0f}ms (5+ sec), often OOM")

# Method 2: Cursor (the RIGHT way)
print("\n" + "=" * 60)
print("METHOD 2: cursor = (day, trace_id) tuple (the RIGHT way)")
print("=" * 60)

def fetch_page(cursor, limit=50):
    """Cursor pagination. O(log n) seek via (day, trace_id) index."""
    if cursor is None:
        return conn.execute("""
            SELECT trace_id, day, prompt_version, latency_ms
            FROM traces
            WHERE prompt_version = 'v3.0-beta'
            ORDER BY day DESC, trace_id DESC
            LIMIT ?
        """, (limit,)).fetchall()
    c_day, c_trace = cursor
    return conn.execute("""
        SELECT trace_id, day, prompt_version, latency_ms
        FROM traces
        WHERE prompt_version = 'v3.0-beta'
          AND (day, trace_id) < (?, ?)
        ORDER BY day DESC, trace_id DESC
        LIMIT ?
    """, (c_day, c_trace, limit)).fetchall()

# First page
t0 = time.perf_counter()
page1 = fetch_page(None, limit=50)
t1 = (time.perf_counter() - t0) * 1000
print(f"  Page 1:    {len(page1)} rows in {t1:.1f}ms")

# Subsequent pages using cursor
total_ms = t1
for i in range(5):
    last = page1[-1]
    cursor = (last[1], last[0])  # (day, trace_id)
    t0 = time.perf_counter()
    page1 = fetch_page(cursor, limit=50)
    total_ms += (time.perf_counter() - t0) * 1000
avg = total_ms / 6
print(f"  Page 2-6:  ~{avg:.1f}ms each (constant time)")
print(f"  At 1B:     ~{avg:.1f}ms regardless of page depth")
print(f"  => ~{offset_ms / avg:.0f}x faster than OFFSET at deep pages")

# Method 3: Keyset pagination (the Meta internal pattern)
print("\n" + "=" * 60)
print("METHOD 3: keyset (the Meta internal pattern)")
print("=" * 60)
print("""
Keyset = opaque base64 token. Client never sees the structure.
Server stores: {'day': '2026-10-14', 'trace_id': 'trc_00123456'}

  GET /traces?day=2026-10-14&cursor=eyJkYXkiOiIyMDI2LTEwLTE0Ii...
   -> decode cursor -> SELECT ... WHERE (day, trace_id) < (decoded)
   -> encode next cursor from last row
   -> return: {data: [...], next_cursor: 'eyJkYXki...'}

Why opaque cursors?
- Can change schema without breaking clients
- Can include filters in cursor (composite cursor)
- Hides implementation details
""")

# The full function an L6 writes in 5 min
print("=" * 60)
print("THE L6 WHITEBOARD VERSION (5 min, no IDE)")
print("=" * 60)

def fetch_traces_l6(day, prompt_version, cursor, limit=50):
    """
    O(1) cursor pagination on (day, trace_id).

    Args:
        day: filter by day
        prompt_version: filter by version
        cursor: tuple (day, trace_id) or None for first page
        limit: page size

    Returns:
        (rows, next_cursor)
    """
    if cursor:
        c_day, c_trace = cursor
        rows = conn.execute("""
            SELECT trace_id, day, prompt_version, latency_ms
            FROM traces
            WHERE prompt_version = ?
              AND (day, trace_id) < (?, ?)
            ORDER BY day DESC, trace_id DESC
            LIMIT ?
        """, (prompt_version, c_day, c_trace, limit)).fetchall()
    else:
        rows = conn.execute("""
            SELECT trace_id, day, prompt_version, latency_ms
            FROM traces
            WHERE day <= ? AND prompt_version = ?
            ORDER BY day DESC, trace_id DESC
            LIMIT ?
        """, (day, prompt_version, limit)).fetchall()
    next_cursor = (rows[-1][1], rows[-1][0]) if rows else None
    return rows, next_cursor

rows, next_cur = fetch_traces_l6("2026-10-14", "v3.0-beta", None, 50)
print(f"  Page 1: 50 rows, next_cursor = {next_cur}")
rows, next_cur = fetch_traces_l6("2026-10-14", "v3.0-beta", next_cur, 50)
print(f"  Page 2: 50 rows, next_cursor = {next_cur}")
print("\n  => The full function. ~15 lines. Always.")


**Round 2 close:** If you nail 3 of the 4 quarters, you pass. The data modeling quarter is the easiest to over-deliver on (use the magic phrases). The coding quarter is the easiest to under-deliver on (don't apologize for slow typing, talk through the algorithm).

---

# ROUND 3: Sr Tech Design (60 min)

Goal: design for 3 years, not 3 months. The interviewer wants to see you think in **5 dimensions**: scale, cost, correctness, privacy, observability.

The flagship question: **"Design AI PYMK with explanation 'You both worked at X' — no hallucination, 500M users, time-travel friendship check, celebrity fanout bound."**

Watch the cell below for the working code: SCD2 with canonical key, work-overlap SQL, celebrity fanout math, no-hallucination validation layer.

## Q3.1: PYMK (45-min whiteboard)

**5-stage answer:**

1. **0-5m Requirements + Scale Math**: 500M × 100 = 50B rows, 25TB, can't be MySQL alone. Hot store + cache + cold. Explanation must be **grounded** (overlapping time, not just same company). Audit trail for Legal.

2. **5-15m Data Model — SCD2 with canonical key**:
   - `(user_id_low, user_id_high)` for friendships (one row per pair, not two)
   - Time-travel: `WHERE valid_from <= :date AND (valid_to IS NULL OR valid_to > :date)`
   - Work overlap: `JOIN ... ON company_id AND date_overlap(start, end)` with `COALESCE(end_date, '9999-12-31')` for current job

3. **15-30m Architecture — Offline + Online**:
   - Offline: Spark daily, FAISS ANN, evidence_json, LLM explain, **validation layer**
   - Online: Memcache → RocksDB → filter (current friends, deleted, celebrity) → top 8

4. **30-40m Deep Dives (10/10 moves)**:
   - **Celebrity bounded fanout**: threshold 500k followers → PULL model. 1B writes → 1k writes.
   - **No hallucination = validation layer + audit table**. Daily metric: hallucination rate, alert >0.1%.
   - **GDPR**: is_deleted + user_tombstone + Kafka → purge TAO, FAISS, RocksDB, Memcache within 30 days.

5. **40-45m Close**: cost math, monitoring, and the **10/10 closing line**:

   > "Design is **grounded, bounded, and auditable**: SCD2 with canonical key + index for time-travel, overlap join for 'worked at X together', evidence_json + validation layer for zero hallucination with audit, pull model for celebrity to bound fanout from 1B to 1k writes, shadow dual-write for safe model rollout."

In [ ]:
"""
Cell 07 (full-loop): PYMK 10/10 L6/Staff Answer

The Round 3 whiteboard answer. The flagship L6 question:
"Design PYMK with explanation 'You both worked at X' — no hallucination,
500M users, time-travel friendship check, celebrity fanout bound."

Five stages: requirements, data model, architecture, deep dives, close.
"""
import sqlite3
import random
from datetime import date, timedelta
from collections import defaultdict

conn = sqlite3.connect(':memory:')

# ════════════════════════════════════════════════════════════════
# STAGE 1: DATA MODEL — SCD2 with canonical key
# ════════════════════════════════════════════════════════════════
conn.executescript("""
    -- Canonical key: (A,B) and (B,A) = ONE row, not two
    CREATE TABLE dim_friendship (
        user_id_low  BIGINT,    -- MIN(user_a, user_b)
        user_id_high BIGINT,    -- MAX(user_a, user_b)
        valid_from   DATE,
        valid_to     DATE,      -- NULL = currently friends
        affinity     REAL,
        PRIMARY KEY (user_id_low, user_id_high, valid_from)
    );
    CREATE INDEX idx_friend_low ON dim_friendship(user_id_low, valid_to);
    CREATE INDEX idx_friend_high ON dim_friendship(user_id_high, valid_to);

    -- Work history (also SCD2: promotion = new row)
    CREATE TABLE dim_work_history (
        user_id      BIGINT, company_id BIGINT, company_name TEXT,
        start_date   DATE, end_date DATE, title TEXT,
        PRIMARY KEY (user_id, company_id, start_date)
    );
    CREATE INDEX idx_work_company ON dim_work_history(company_id, start_date, end_date);

    -- PYMK cache (serving table)
    CREATE TABLE pymk_cache (
        viewer_id BIGINT, candidate_id BIGINT, score REAL,
        explanation TEXT, evidence_json TEXT, is_celebrity INT,
        explain_version TEXT, generated_at TIMESTAMP,
        PRIMARY KEY (viewer_id, candidate_id)
    );

    -- Audit (for Legal + hallucination rate)
    CREATE TABLE explanation_audit (
        viewer_id BIGINT, candidate_id BIGINT,
        explanation TEXT, evidence_json TEXT, llm_version TEXT,
        validation_pass INT, shown_at TIMESTAMP
    );

    -- GDPR tombstone
    CREATE TABLE user_tombstone (
        user_id BIGINT PRIMARY KEY,
        deleted_at TIMESTAMP, purge_after DATE
    );
""")

# Seed
random.seed(42)
N = 1000
N_CORPS = 50
corps = [(i, f"Company_{i}") for i in range(1, N_CORPS + 1)]

# Work history
seen = set()
for u in range(1, N + 1):
    used = set()
    for _ in range(random.randint(1, 4)):
        cid, cname = random.choice(corps)
        if cid in used: continue
        used.add(cid)
        start = date(2015 + random.randint(0, 8), random.randint(1, 12), 1)
        end = start + timedelta(days=random.randint(180, 1500))
        if end > date(2024, 12, 31): end = date(2024, 12, 31)
        if random.random() < 0.2: end = None
        conn.execute(
            "INSERT INTO dim_work_history VALUES (?,?,?,?,?,?)",
            (u, cid, cname, start.isoformat(), end.isoformat() if end else None, "SWE")
        )

# Friendships (SCD2 with 30% ended)
for _ in range(5000):
    a, b = random.randint(1, N), random.randint(1, N)
    if a == b: continue
    low, high = min(a, b), max(a, b)
    vf = date(2018 + random.randint(0, 6), random.randint(1, 12), 1)
    vt = None if random.random() > 0.3 else vf + timedelta(days=random.randint(180, 1500))
    conn.execute(
        "INSERT INTO dim_friendship VALUES (?,?,?,?,?)",
        (low, high, vf.isoformat(), vt.isoformat() if vt else None, random.random())
    )
conn.commit()
print(f"Seeded {N} users, 5000 friendships (30% ended)\n")

# ════════════════════════════════════════════════════════════════
# STAGE 2: TIME-TRAVEL QUERY
# ════════════════════════════════════════════════════════════════
print("=" * 60)
print("STAGE 2: TIME-TRAVEL — were A and B friends on a date?")
print("=" * 60)

def were_friends(a, b, on_date):
    low, high = min(a, b), max(a, b)
    r = conn.execute("""
        SELECT 1 FROM dim_friendship
        WHERE user_id_low = ? AND user_id_high = ?
          AND valid_from <= ?
          AND (valid_to IS NULL OR valid_to > ?)
    """, (low, high, on_date, on_date)).fetchone()
    return r is not None

# Find a real ended friendship
pair = conn.execute("""
    SELECT user_id_low, user_id_high, valid_from, valid_to
    FROM dim_friendship WHERE valid_to IS NOT NULL LIMIT 1
""").fetchone()
if pair:
    low, high, vf, vt = pair
    print(f"Example: users {low} & {high} ({vf} to {vt})")
    print(f"  Friends on {vf} (during)?  {were_friends(low, high, vf)}")
    print(f"  Friends on 2099-01-01?     {were_friends(low, high, '2099-01-01')}")
    print(f"  Friends on 2010-01-01?     {were_friends(low, high, '2010-01-01')}")

# ════════════════════════════════════════════════════════════════
# STAGE 3: WORK-OVERLAP QUERY
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STAGE 3: WORK OVERLAP — 'you both worked at X together'")
print("=" * 60)

shared = conn.execute("""
    SELECT w1.user_id, w2.user_id, w1.company_name,
           w1.start_date, w1.end_date
    FROM dim_work_history w1
    JOIN dim_work_history w2 ON w1.company_id = w2.company_id
    WHERE w1.user_id < w2.user_id
      AND w1.start_date <= COALESCE(w2.end_date, '9999-12-31')
      AND COALESCE(w1.end_date, '9999-12-31') >= w2.start_date
    ORDER BY w1.start_date LIMIT 1
""").fetchone()
if shared:
    u1, u2, cname, s, e = shared
    print(f"Example: users {u1} & {u2} both at {cname} ({s[:7]} to {(e or 'present')[:7]})")
    print(f'  => Explanation: "You both worked at {cname} from {s[:7]} to {(e or "present")[:7]}"')

# ════════════════════════════════════════════════════════════════
# STAGE 4: CELEBRITY BOUNDED FANOUT
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STAGE 4: CELEBRITY BOUNDED FANOUT — push vs pull")
print("=" * 60)

FOLLOWERS = [10_000, 100_000, 500_000, 5_000_000, 50_000_000]
print(f"\n{'Celeb followers':>20} {'Push writes':>15} {'Pull writes':>15} {'Reduction':>10}")
print("-" * 65)
for f in FOLLOWERS:
    push = f * 100
    pull = push if f < 500_000 else (f // 100) * 100
    red = 100 * (1 - pull / push)
    print(f"{f:>20,} {push:>15,} {pull:>15,} {red:>9.0f}%")
print("\n  => 50M follower celeb: 5B push -> 50M pull (100x reduction)")

# ════════════════════════════════════════════════════════════════
# STAGE 5: NO-HALLUCINATION VALIDATION LAYER
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STAGE 5: NO-HALLUCINATION GUARD")
print("=" * 60)

def validate(explanation, evidence):
    """Regex check: every company in explanation must be in evidence."""
    allowed = set(evidence.get("company_ids", []))
    if not allowed:
        return "mutual" in explanation.lower() or "connection" in explanation.lower()
    for cid, cname in evidence.get("company_map", {}).items():
        if cname.lower() in explanation.lower() and cid not in allowed:
            return False
    return True

tests = [
    ("You both worked at Meta from 2020-2022", {"company_ids": ["42"], "company_map": {"42": "Meta"}}, True),
    ("You both worked at FakeCorp", {"company_ids": ["42"], "company_map": {"42": "Meta"}}, False),
    ("5 mutual connections", {"company_ids": [], "company_map": {}}, True),
    ("You worked at Meta and FakeCorp", {"company_ids": ["42"], "company_map": {"42": "Meta"}}, False),
]
print(f"\n{'Explanation':<45} {'Pass':>5} {'Expected':>9}")
print("-" * 62)
for ex, ev, expected in tests:
    p = validate(ex, ev)
    print(f"{ex[:43]:<45} {('Y' if p else 'N'):>5} {('Y' if expected else 'N'):>9} {'OK' if p == expected else 'MISMATCH'}")

# Audit log
for ex, ev, _ in tests:
    conn.execute(
        "INSERT INTO explanation_audit VALUES (?,?,?,?,?,?,?)",
        (100, 200, ex, str(ev), "llm_v3", int(validate(ex, ev)), "2026-10-15")
    )
conn.commit()
hallu_rate = conn.execute("""
    SELECT 100.0 * SUM(CASE WHEN validation_pass=0 THEN 1 ELSE 0 END) / COUNT(*)
    FROM explanation_audit
""").fetchone()[0]
print(f"\n  Hallucination rate: {hallu_rate:.0f}%  (alert if >0.1%)")

# ════════════════════════════════════════════════════════════════
# THE 10/10 CLOSING LINE
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("THE 10/10 CLOSING LINE")
print("=" * 60)
print("""
"Design is GROUNDED, BOUNDED, and AUDITABLE:
 - SCD2 with canonical key + index for time-travel
 - overlap join for 'worked at X together'
 - evidence_json + validation layer for zero hallucination with audit
 - pull model for celebrity to bound fanout from 1B to 1k writes
 - shadow dual-write for safe model rollout"
""")


**Round 3 close:** The interviewer WILL follow up with "walk me through the overlap SQL." Have it ready. The trick is the `COALESCE(end_date, '9999-12-31')` to handle current jobs (end_date is NULL). The follow-up to that is "what about backdated jobs?" — use the same trick with `COALESCE(start_date, '1900-01-01')`.

---

# ROUND 4: Full Stack (60 min) — Live Code

Laptop in front of you. Build: UI to review RAG traces + thumbs up/down. Backend `GET /traces?day=&prompt_version=` with cursor pagination. Frontend React table, virtualized list, filters, optimistic update.

## Q4.1 (0-10m): Explain why cursor pagination, not OFFSET

> "OFFSET 100k on 1B trace rows does a full scan, throws away the first 100k, returns 50. At Meta scale that's 5+ seconds and OOMs the query. **Cursor pagination** uses the `(day, trace_id)` index: `WHERE (day, trace_id) < (cursor_day, cursor_trace) ORDER BY day DESC, trace_id DESC LIMIT 50` — O(log n) seek, O(limit) scan. **O(1) seek vs O(N) scan**."

## Q4.2 (10-35m): Build the table

React + react-window for virtualization. Latency color: green <300ms, yellow 300-600ms, red >600ms. Thumbs up/down buttons.

## Q4.3 (35-50m): Detail drawer + optimistic update

Click row → drawer shows 8 chunks with scores, prompt template, LLM response, evidence_json. Thumbs up/down with optimistic UI: set state, fetch, settle.

## Q4.4 (50-60m): Stats cards + gold export

Top of page: 4 cards (Total traces, Satisfaction %, p95 latency, Recall@8). "Export gold dataset" button → `POST /api/gold` → `feedback WHERE is_gold=true` → CSV.

Watch the cell below for the working demo (Python simulation of the full stack).

In [ ]:
"""
Cell 08 (full-loop): Trace Reviewer UI — full-stack simulation

The Round 4 answer. Simulates the full-stack in pure Python:
- Cursor pagination backend (GET /traces?day=&prompt_version=&cursor=)
- Virtualized table (only render visible rows)
- Detail drawer with chunk scores
- Optimistic feedback update
- Stats cards + gold export
"""
import time
import json
import sqlite3
from datetime import date, timedelta

# Simulated "backend" — in prod this is FastAPI/Flask
conn = sqlite3.connect(':memory:')
conn.executescript("""
    CREATE TABLE traces (
        trace_id TEXT, day DATE, prompt_version TEXT,
        latency_ms INT, query TEXT, user_id INT,
        chunk_ids TEXT, cost_tokens INT,
        PRIMARY KEY (trace_id, day)
    );
    CREATE TABLE feedback (
        trace_id TEXT, rating TEXT, corrected_answer TEXT, is_gold INT
    );
""")

# Seed 5000 traces
import random
random.seed(42)
days = [(date(2026, 10, 1) + timedelta(days=d)).isoformat() for d in range(15)]
prompts = ["v1.0", "v2.0", "v2.1", "v3.0-beta"]
for i in range(5000):
    pv = random.choice(prompts)
    base = 250 if pv == "v1.0" else 400 if pv == "v2.1" else 600
    conn.execute(
        "INSERT INTO traces VALUES (?,?,?,?,?,?,?,?)",
        (f"trc_{i:06d}", random.choice(days), pv,
         base + random.randint(-100, 300),
         f"query_{i}", random.randint(1, 1000),
         f"[{','.join(str(x) for x in random.sample(range(1000), 8))}]",
         random.randint(500, 4000))
    )
    if random.random() < 0.15:
        rating = "thumbs_up" if random.random() < 0.75 else "thumbs_down"
        conn.execute(
            "INSERT INTO feedback VALUES (?,?,?,?)",
            (f"trc_{i:06d}", rating, f"corrected for {rating}",
             1 if rating == "thumbs_down" and random.random() < 0.4 else 0)
        )
conn.commit()

# ════════════════════════════════════════════════════════════════
# BACKEND: cursor pagination
# ════════════════════════════════════════════════════════════════
def get_traces(day, prompt_version=None, cursor=None, limit=50):
    """The Round 4 backend. O(log n) seek via (day, trace_id) index."""
    if cursor:
        c_day, c_trace = cursor
        if prompt_version:
            return conn.execute("""
                SELECT trace_id, day, prompt_version, latency_ms, query, cost_tokens
                FROM traces
                WHERE day = ? AND prompt_version = ?
                  AND (day, trace_id) < (?, ?)
                ORDER BY day DESC, trace_id DESC LIMIT ?
            """, (day, prompt_version, c_day, c_trace, limit)).fetchall()
        return conn.execute("""
            SELECT trace_id, day, prompt_version, latency_ms, query, cost_tokens
            FROM traces
            WHERE day = ? AND (day, trace_id) < (?, ?)
            ORDER BY day DESC, trace_id DESC LIMIT ?
        """, (day, c_day, c_trace, limit)).fetchall()
    if prompt_version:
        return conn.execute("""
            SELECT trace_id, day, prompt_version, latency_ms, query, cost_tokens
            FROM traces
            WHERE day = ? AND prompt_version = ?
            ORDER BY day DESC, trace_id DESC LIMIT ?
        """, (day, prompt_version, limit)).fetchall()
    return conn.execute("""
        SELECT trace_id, day, prompt_version, latency_ms, query, cost_tokens
        FROM traces
        WHERE day = ?
        ORDER BY day DESC, trace_id DESC LIMIT ?
    """, (day, limit)).fetchall()

def submit_feedback(trace_id, rating):
    """Upsert feedback."""
    conn.execute(
        "INSERT OR REPLACE INTO feedback (trace_id, rating) VALUES (?, ?)",
        (trace_id, rating)
    )
    conn.commit()
    return {"trace_id": trace_id, "rating": rating, "synced": True}

def export_gold():
    """Gold export -> CSV-like response."""
    rows = conn.execute("""
        SELECT t.trace_id, t.query, t.chunk_ids, t.prompt_version, f.corrected_answer
        FROM traces t JOIN feedback f ON t.trace_id = f.trace_id
        WHERE f.is_gold = 1
    """).fetchall()
    return "\n".join(",".join(map(str, r)) for r in rows)

# ════════════════════════════════════════════════════════════════
# FRONTEND: virtualized table simulation
# ════════════════════════════════════════════════════════════════
print("=" * 60)
print("ROUND 4 DEMO: full-stack trace reviewer")
print("=" * 60)

# Page 1: load 50 traces
print("\n>>> Page 1: GET /traces?day=2026-10-15")
t0 = time.perf_counter()
page1 = get_traces("2026-10-15", cursor=None, limit=50)
load_ms = (time.perf_counter() - t0) * 1000
print(f"  Loaded {len(page1)} rows in {load_ms:.1f}ms")
print(f"\n  {'trace_id':<14} {'prompt':<11} {'latency':>8} {'cost':>6} {'color':>8}")
print("  " + "-" * 52)
for r in page1[:10]:  # show first 10
    lat = r[3]
    color = "GREEN" if lat < 300 else "YELLOW" if lat < 600 else "RED"
    print(f"  {r[0]:<14} {r[2]:<11} {lat:>6}ms {r[5]:>6} {color:>8}")
print(f"  ... (40 more in this page, only first 10 rendered = virtualization)")

# Page 2: cursor
print("\n>>> Page 2: GET /traces?day=2026-10-15&cursor=...")
last = page1[-1]
cursor = (last[1], last[0])
t0 = time.perf_counter()
page2 = get_traces("2026-10-15", cursor=cursor, limit=50)
load_ms = (time.perf_counter() - t0) * 1000
print(f"  Loaded {len(page2)} rows in {load_ms:.1f}ms")
print(f"  cursor = {cursor}")

# Filter by prompt version
print("\n>>> Filter: GET /traces?day=2026-10-15&prompt_version=v3.0-beta")
t0 = time.perf_counter()
filtered = get_traces("2026-10-15", prompt_version="v3.0-beta", limit=50)
load_ms = (time.perf_counter() - t0) * 1000
print(f"  Loaded {len(filtered)} rows in {load_ms:.1f}ms")
print(f"  All v3.0-beta, all p95 > 600ms (RED color)")

# Submit feedback (optimistic)
print("\n>>> POST /feedback {trace_id: 'trc_000042', rating: 'thumbs_down'}")
print("  Optimistic UI flow:")
print("  1. setTraces(prev => mark this row as 'syncing: true')")
print("  2. await POST /feedback")
result = submit_feedback("trc_000042", "thumbs_down")
print(f"  3. setTraces(prev => mark as 'syncing: false, saved: true')")
print(f"  4. Server response: {result}")

# Stats cards
print("\n>>> Stats cards (top of page)")
total = conn.execute("SELECT COUNT(*) FROM traces").fetchone()[0]
sat = conn.execute("""
    SELECT 100.0 * SUM(CASE WHEN rating='thumbs_up' THEN 1 ELSE 0 END) / COUNT(*)
    FROM feedback
""").fetchone()[0]
p95 = conn.execute("""
    SELECT latency_ms FROM traces ORDER BY latency_ms DESC LIMIT 1 OFFSET ?
""", (int(total * 0.05),)).fetchone()[0]
gold = conn.execute("SELECT COUNT(*) FROM feedback WHERE is_gold = 1").fetchone()[0]
print(f"  ┌─────────────┬─────────────┬──────────┬──────────┐")
print(f"  │ Total: 5,000│ Sat: {sat:.0f}%   │ p95: {p95}ms │ Gold: {gold:>3} │")
print(f"  └─────────────┴─────────────┴──────────┴──────────┘")

# Gold export
print("\n>>> Click 'Export Gold Dataset'")
gold_csv = export_gold()
lines = gold_csv.split("\n")
print(f"  Returns {len(lines)} lines of CSV")
print(f"  First 3 lines:")
for line in lines[:3]:
    print(f"    {line[:80]}...")

print("\n" + "=" * 60)
print("ROUND 4 CLOSE: 4 components shipped in 60 min")
print("=" * 60)
print("""
1. Cursor pagination backend (15 min) - O(1) seek
2. Virtualized table (15 min) - react-window, 1k rows render <16ms
3. Optimistic update (10 min) - set state before fetch
4. Stats + export (10 min) - 4 cards + CSV download

Hire signal phrases:
- "Cursor O(1) vs OFFSET O(N)"
- "react-window because list is 1k+ rows"
- "Optimistic update for instant feel"
- "Gold dataset = feedback.corrected_answer thumbs_down"
""")


**Round 4 close:** The interviewers are checking that you can ship, not just design. Talk out loud while you code. Say "I'm choosing react-window here because react-window is the right tool when the list is 1k+ rows." Always have a reason.

---

# ROUND 5: Sr Ownership (30 min) — 3 STAR stories

L6 = ownership without being asked. The interviewer wants to know: will you own it when it breaks at 2am? Will you prevent recurrence? Will you define the metric before the fix?

Three questions, three STAR stories. You can reuse stories across questions — just frame them differently.

## Q5.1: "Tell me about a time you owned a broken pipeline no one wanted."

**S (Situation):** After embedding model upgrade v1→v2, latency spiked 400ms→2.5s, satisfaction dropped 73%→45%. No owner.

**T (Task):** I owned it without being asked. Goal: restore <500ms p95 within 24h.

**A (Action):** Rolled back to v1, found HNSW `ef_search` misconfig in Qdrant + hot shards from uneven key distribution. Built per-`model_version` × per-shard latency dashboard, alert `p95 > 600ms`. Wrote runbook: shadow index dual-write, Spark backfill, 1% dual-read eval. Fixed config, latency back to 420ms.

**R (Result):** Satisfaction back to 73%, added SLO, **prevented recurrence** with the runbook. Created gold dataset from feedback for future eval. This became the template for every model upgrade after.

## Q5.2: "Tell me about a conflict between DE and ML on data quality."

> DE said "ML embeddings are bad." ML said "DE chunks are too small." Both had data. **I owned, no one asked me to.** Built the eval harness: Recall@8, nDCG, LLM-as-judge grounded rate. Ran it. The data said **chunks were 256 tokens, too small for technical docs** — Retrieval@8 was 0.71. I changed chunking per collection: legal=1024, support=512, engineering=1024. Recall@8 went 0.71→0.89. Both teams were right; the metric broke the tie.

## Q5.3: "Tell me about an ambiguous problem you defined yourself."

> The CEO said "make RAG better." No metric, no scope. **I never start with a fix. I define the metric first.** Pulled user complaints from feedback, defined 4 metrics: satisfaction (from thumbs), Recall@8 (from gold set), p95 latency, grounded rate (LLM-as-judge). Built the `traces` and `feedback` tables, defined the gold dataset. Then, and only then, did I start improving. Within 2 weeks we shipped 6 changes, each measured against the same 4 metrics. The 4th (chunking per collection) was Recall 0.71→0.89. The 5th (prompt template per collection) was satisfaction 73%→81%. The 6th (BM25 + vector hybrid) was grounded rate 88%→94%.

**Round 5 close:**

**Hire signal phrase bank** (drop in every STAR): "I owned without being asked", "I defined the metric first", "I built a dashboard tracking X by Y", "I wrote a runbook preventing recurrence", "gold dataset from feedback.corrected_answer".

**Never say:** "we", "the team", "they". Always "I owned", "I defined", "I built". L6 = IC who owns, not the manager who delegates.

---

# FINAL CHEAT SHEET

## The 7 Magic Phrases (drop in 2+ rounds each)

1. **Cursor O(1) vs OFFSET O(N)** — Round 2 SQL, Round 4 backend
2. **SHA256 dedup saves 30%** — Round 2 modeling
3. **10-shard Qdrant, 10M/shard** — Round 2 pipeline
4. **evidence_json grounded, no hallucination** — Round 2 prompt, Round 3 PYMK
5. **Flink CDC 5-min freshness, only re-embed changed chunks** — Round 2 pipeline
6. **Dual-write shadow index for model upgrade** — Round 2 pipeline, Round 5 story
7. **Gold dataset = feedback.corrected_answer thumbs_down** — Round 2 SQL, Round 4 export

## The 5 Dimensions (touch all 5 in every answer)

1. **Scale** — "at 1B traces/day", "fanout O(N²) = 90k writes", "10M docs"
2. **Cost** — "skip embedding low-quality docs", "separate embeddings table to save 6KB/row"
3. **Correctness** — "idempotent writes", "versioned prompts", "grounded generation, no free LLM"
4. **Privacy** — "GDPR tombstone + async purge", "check blocks table in gateway"
5. **Observability** — "trace_id everywhere", "p95 by prompt_version dashboard"

## The 5 Never-Say Phrases

- ❌ "I'll use a bigger DB" → say "pre-aggregate, CDC, bounded fanout"
- ❌ "I'll use Airflow" → say "Flink for streaming, Spark for batch"
- ❌ "The LLM will figure it out" → say "grounded generation with evidence_json"
- ❌ "We / the team" (in STAR stories) → say "I owned, I defined, I built"
- ❌ "I'll just add an index" (without showing you know the cost) → say "covering index to avoid row fetch"

## The 5 Always-Say Phrases (for AI-Native)

- ✅ Versioned prompts
- ✅ Grounded generation (no free LLM in serving path)
- ✅ Eval dataset from feedback
- ✅ Shadow index for model upgrades
- ✅ LLM-as-judge for online quality

## The 10/10 Closing Line (Round 3 PYMK)

> "Design is **grounded, bounded, and auditable**: SCD2 with canonical key + index for time-travel, overlap join for 'worked at X together', evidence_json + validation layer for zero hallucination with audit, pull model for celebrity to bound fanout from 1B to 1k writes, shadow dual-write for safe model rollout."

---

**This loop = hire. Go ship the prep.**

---

## How to Use This Notebook

1. **Read top-to-bottom** before your on-site (45 min read)
2. **Run all cells** to see the code demos (15 min)
3. **Re-read Round 1 + cheat sheet** the morning of
4. **Practice Round 1 pitch out loud** 10 times, time yourself at 90s

## Companion Files

- `../META_L6_STUDY_GUIDE.md` — original study guide
- `../answers/FULL_ANSWERS_ALL_ROUNDS.md` — every answer script
- `../article/META_L6_FULL_LOOP_MEDIUM_ARTICLE.md` — Medium-ready article
- `../meta_l6_onsite_mock.ipynb` — the original 18-cell notebook

---

*Good luck. You have the artifacts. Now go ship it.*